# 🧪 Praktikum 11 – Modell-Serving & API-Architekturen
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Custom Modell-Konfigurationen via Modelfile · Einen REST-API Wrapper mit FastAPI bauen · Performance vs. Quantisierung evaluieren

⏱️ **Dauer:** 90 Minuten

In [ ]:
import os, sys, subprocess, shutil, time, requests
MODEL = "qwen3.5:0.8b"
def install_if_missing(pkgs):
    cmd = [sys.executable, "-m", "pip", "install"] + (["--break-system-packages"] if sys.prefix == getattr(sys, "base_prefix", sys.prefix) else [])
    subprocess.check_call(cmd + pkgs)
install_if_missing(["ollama", "requests", "fastapi", "uvicorn"])
import ollama
print("✅ Setup abgeschlossen.")

## Teil 1 – Deep Dive: Modelfiles ⏱️ 35 min
Wir konfigurieren Context-Window, Stop-Tokens und System-Prompts.

In [ ]:
custom_config = f"""
FROM {MODEL}
SYSTEM Du bist ein strenger Python-Code-Reviewer. Antworte NUR mit Code-Korrekturen.
PARAMETER num_ctx 4096
PARAMETER stop "---End---"
PARAMETER temperature 0.2
"""

with open("ReviewerModel", "w") as f: f.write(custom_config)

print("Registriere Custom Model...")
subprocess.check_call(["ollama", "create", "code-reviewer", "-f", "ReviewerModel"])

test_code = "def add(a, b): return a+b # No documentation!"
res = ollama.chat(model="code-reviewer", messages=[{"role": "user", "content": test_code}])
print(f"Reviewer Antwort:\n{res['message']['content']}")

## Teil 2 – Modell-Serving via FastAPI (Konzept) ⏱️ 30 min
Wir bauen einen Wrapper, der Anfragen vorverarbeitet.

In [ ]:
from fastapi import FastAPI
import threading

app = FastAPI()

@app.get("/generate")
def generate(prompt: str):
    # Hier könnte Logging oder PII-Filterung passieren
    res = ollama.generate(model=MODEL, prompt=prompt)
    return {"response": res['response']}

print("FastAPI Server Instanz erstellt. (In Produktion via uvicorn.run)")

## Teil 3 – Quantization Impact Math ⏱️ 25 min
Berechne die Speicherersparnis und diskutiere den Qualitätsverlust.

In [ ]:
def calc_mem(params_b, bits):
    return (params_b * bits) / 8

model_sizes = [0.8, 7, 70]
bit_depths = [16, 8, 4, 2]

for p in model_sizes:
    print(f"--- {p}B Modell ---")
    for b in bit_depths:
        print(f"{b}-bit: {calc_mem(p, b):.2f} GB RAM")

## 🧩 Aufgaben
1. **Custom Modelfile:** Erstelle ein Modell mit einem extrem kleinen Context Window (`num_ctx 10`). Was passiert bei langen Fragen?
2. **API Wrapper:** Erweitere die FastAPI Route so, dass sie zusätzlich den Token-Verbrauch (via Ollama Response) zurückgibt.
3. **Benchmarking:** Vergleiche die Zeit für 'First Token' (TTFT) bei verschiedenen `temperature` Einstellungen.